In [6]:
import pandas as pd
import cv2
import matplotlib.pyplot
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

In [8]:
dataset_dir = r"E:\Kuliah\comvision\projek\asl_dataset"
img_size = (64, 64)

In [9]:
X = []
y = []
for label in sorted(os.listdir(dataset_dir)):
    folder = os.path.join(dataset_dir, label)
    if not os.path.isdir(folder):
        continue
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        for fp in glob.glob(os.path.join(folder, ext)):
            img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, img_size)
            X.append(img.flatten())
            y.append(label)

X = np.array(X)
y = np.array(y)
print("Loaded", X.shape, "samples with labels:", np.unique(y).shape[0])

Loaded (2515, 4096) samples with labels: 36


In [10]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [11]:
param_grid = {
    "C": [1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}
svc = SVC()
grid = GridSearchCV(svc, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(X_train_s, y_train)

best = grid.best_estimator_
print("Best params:", grid.best_params_)

y_pred = best.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Accuracy: 0.9840954274353877
              precision    recall  f1-score   support

           0       1.00      0.86      0.92        14
           1       0.93      1.00      0.97        14
           2       1.00      1.00      1.00        14
           3       1.00      1.00      1.00        14
           4       1.00      0.93      0.96        14
           5       0.93      1.00      0.97        14
           6       0.93      1.00      0.97        14
           7       1.00      1.00      1.00        14
           8       1.00      1.00      1.00        14
           9       1.00      1.00      1.00        14
           a       1.00      1.00      1.00        14
           b       1.00      1.00      1.00        14
           c       1.00      1.00      1.00        14
           d       1.00      1.00      1.00        14
           e       1.00      1.00      1

In [12]:
from sklearn.model_selection import cross_val_score

print("Train acc:", best.score(X_train_s, y_train))
print("Test acc: ", best.score(X_test_s, y_test))

X_full = np.vstack((X_train, X_test))
y_full = np.hstack((y_train, y_test))
X_full_s = scaler.transform(X_full)

svc_for_cv = SVC(**best.get_params())
cv_scores = cross_val_score(svc_for_cv, X_full_s, y_full, cv=5, n_jobs=-1)
print("5-fold CV acc: %.4f ± %.4f" % (cv_scores.mean(), cv_scores.std()))

Train acc: 1.0
Test acc:  0.9840954274353877
5-fold CV acc: 0.9746 ± 0.0058


In [13]:
joblib.dump({"model": best, "scaler": scaler, "label_encoder": le}, r"E:\Kuliah\comvision\projek\svm_asl_model.joblib")

def predict_image(path):
    data = joblib.load(r"E:\Kuliah\comvision\projek\svm_asl_model.joblib")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, img_size).flatten().reshape(1, -1)
    img_s = data["scaler"].transform(img)
    pred = data["model"].predict(img_s)
    return data["label_encoder"].inverse_transform(pred)[0]
